# Benchmark: representation × metric grid

Answers **Reviewer 1 points 2, 3 and 5** for the Molecular Informatics
revision by running every combination of

- **Representation**: ECFP4 (1k / 2k / 4k bits), ECFP6, MACCS (166 bits),
  RDKit path fingerprint, RDKit physicochemical descriptors (∼200 features),
  Mol2Vec (300D), Uni-Mol (512D), hybrid (812D)
- **Metric**: Tanimoto, Tversky (α=0.8/β=0.2 and α=0.2/β=0.8),
  Dice, Cosine, Mahalanobis distance d$_M$, Mahalanobis angle $\theta_M$

against the ground-truth property proximity test used in
`benchmark.ipynb` — "do the molecules MSI calls similar *actually have*
similar measured properties?"

Covers:

| Reviewer point | Answer in this notebook |
|---|---|
| **2** — mention Tversky as a generalisation of Tanimoto | Tversky (two asymmetric variants) is in the metric axis |
| **3** — representation choice matters as much as the metric | The grid makes the representation effect visible as a separate axis |
| **5** — benchmark against a wider range of representations | All three families are present (fingerprints varied, descriptors, embeddings) |

The outputs of this notebook — win-rate tables and heatmaps — are the
evidence that goes into the author response letter.

## Section 0 — Imports and configuration

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from gensim.models import Word2Vec

from benchmark_matrix import (
    DatasetCase, run_grid,
    win_rate_table, summary_by_family,
)
from representations import ALL_REPRESENTATIONS
from similarity_measures import METRIC_ORIENTATION, COMPATIBLE_METRICS

sns.set_theme(style='white', font_scale=1.05)

DATA_DIR          = 'data'
BENCHMARK_DIR     = os.path.join(DATA_DIR, 'benchmark')
MOL2VEC_OUTPUT    = 'output_benchmark'   # results from benchmark.ipynb
UNIMOL_OUTPUT     = 'output_unimol'      # results from benchmark_unimol.ipynb
REP_CACHE         = 'output_representations'
RESULTS_DIR       = 'output_grid'
os.makedirs(RESULTS_DIR, exist_ok=True)

N_BOOT     = 2_000   # bootstrap iterations for the random-baseline p-value
RANDOM_SEED = 42

MODEL_PATH = os.path.join(DATA_DIR, 'model_300dim.pkl')
print('Loading mol2vec model…')
t0 = time.time()
mol2vec_model = Word2Vec.load(MODEL_PATH)
print(f'  loaded in {time.time()-t0:.1f}s '
      f'(vocab {len(mol2vec_model.wv):,}, dim {mol2vec_model.wv.vector_size})')

## Section 1 — Dataset cases

Same (dataset, reference-molecule) pairs as `benchmark.ipynb`, now also
including any larger/complex reference that `prepare_benchmark.py` has
produced (Reviewer 1 point 4). Each case points at a CSV with the
reference molecule in row 0 and one or more ground-truth property columns.

In [ ]:
CASES = [
    # --- QM9 / electronic properties ---
    DatasetCase(name='qm9_aniline',
                filename=os.path.join(DATA_DIR, 'qm9_anilin.csv'),
                properties=['gap', 'homo', 'lumo', 'mu', 'alpha'],
                n_values=[10, 25, 50, 100, 250],
                dataset_label='QM9', reference_label='aniline'),
    DatasetCase(name='qm9_pyridine',
                filename=os.path.join(BENCHMARK_DIR, 'qm9_pyridine.csv'),
                properties=['gap', 'homo', 'lumo', 'mu'],
                n_values=[10, 25, 50, 100, 250],
                dataset_label='QM9', reference_label='pyridine'),

    # --- QM8 / excited-state spectra ---
    DatasetCase(name='qm8_aniline',
                filename=os.path.join(BENCHMARK_DIR, 'qm8_aniline.csv'),
                properties=['E1-CC2', 'E2-CC2', 'E1-CAM'],
                n_values=[10, 25, 50, 100, 250],
                dataset_label='QM8', reference_label='aniline'),
    DatasetCase(name='qm8_pyridine',
                filename=os.path.join(BENCHMARK_DIR, 'qm8_pyridine.csv'),
                properties=['E1-CC2', 'E2-CC2', 'E1-CAM'],
                n_values=[10, 25, 50, 100, 250],
                dataset_label='QM8', reference_label='pyridine'),
    DatasetCase(name='qm8_phenol',
                filename=os.path.join(BENCHMARK_DIR, 'qm8_phenol.csv'),
                properties=['E1-CC2', 'E2-CC2', 'E1-CAM'],
                n_values=[10, 25, 50, 100, 250],
                dataset_label='QM8', reference_label='phenol'),

    # --- ESOL / aqueous solubility ---
    DatasetCase(name='esol_caffeine',
                filename=os.path.join(BENCHMARK_DIR, 'esol_caffeine.csv'),
                properties=['logS'],
                n_values=[10, 25, 50, 100],
                dataset_label='ESOL', reference_label='caffeine'),
    DatasetCase(name='esol_naphthalene',
                filename=os.path.join(BENCHMARK_DIR, 'esol_naphthalene.csv'),
                properties=['logS'],
                n_values=[10, 25, 50, 100],
                dataset_label='ESOL', reference_label='naphthalene'),

    # --- FreeSolv / hydration free energy ---
    DatasetCase(name='freesolv_toluene',
                filename=os.path.join(BENCHMARK_DIR, 'freesolv_toluene.csv'),
                properties=['hydration_free_energy'],
                n_values=[10, 25, 50],
                dataset_label='FreeSolv', reference_label='toluene'),
    DatasetCase(name='freesolv_ethanol',
                filename=os.path.join(BENCHMARK_DIR, 'freesolv_ethanol.csv'),
                properties=['hydration_free_energy'],
                n_values=[10, 25, 50],
                dataset_label='FreeSolv', reference_label='ethanol'),

    # --- Lipophilicity / drug-like complex molecules (point 4) ---
    DatasetCase(name='lipo_diclofenac',
                filename=os.path.join(BENCHMARK_DIR, 'lipo_diclofenac.csv'),
                properties=['logD'],
                n_values=[10, 25, 50, 100],
                dataset_label='Lipophilicity', reference_label='diclofenac'),
    DatasetCase(name='lipo_naproxen',
                filename=os.path.join(BENCHMARK_DIR, 'lipo_naproxen.csv'),
                properties=['logD'],
                n_values=[10, 25, 50, 100],
                dataset_label='Lipophilicity', reference_label='naproxen'),
]

# Auto-include any extra large/complex references produced by prepare_benchmark.py
# after the initial scan (celecoxib, sildenafil, losartan, atorvastatin, …).
_complex_names = ('celecoxib', 'sildenafil', 'losartan', 'atorvastatin',
                  'simvastatin', 'tamoxifen', 'fluoxetine', 'imipramine')
for ref in _complex_names:
    path = os.path.join(BENCHMARK_DIR, f'lipo_{ref}.csv')
    if os.path.exists(path):
        CASES.append(DatasetCase(
            name=f'lipo_{ref}', filename=path,
            properties=['logD'], n_values=[10, 25, 50, 100],
            dataset_label='Lipophilicity', reference_label=ref,
        ))

print(f'Configured {len(CASES)} (dataset, reference) cases:')
for c in CASES:
    n_rows = sum(1 for _ in open(c.filename)) - 1 if os.path.exists(c.filename) else 0
    print(f'  {c.dataset_label:>14s}/{c.reference_label:<14s}  '
          f'({n_rows:>7,} mol)  properties={c.properties}')

## Section 2 — Representation list and compute-cost tiering

Split representations into **cheap** (seconds per dataset) and **expensive**
(minutes/hours) so the notebook can be run incrementally:

- **Tier 1** (cheap, on-the-fly): fingerprints + RDKit descriptors + Mol2Vec.
  These run end-to-end in minutes on CPU.
- **Tier 2** (expensive, cached): Uni-Mol and hybrid. Reuse the vectors
  already produced by `benchmark_unimol.ipynb` in `output_unimol/`.

In [ ]:
REPS_TIER_1 = [
    # fingerprint family — varying length and design (point 5)
    'ecfp4_1024', 'ecfp4_2048', 'ecfp4_4096', 'ecfp6_2048',
    'maccs', 'rdkit_fp',
    # descriptor family (point 5: physicochemical descriptors)
    'rdkit_physchem',
    # embedding family — semantic / distributional (paper's primary)
    'mol2vec',
]
REPS_TIER_2 = ['unimol', 'hybrid']  # needs pre-computed 512D CSVs

RUN_TIER_2 = True   # flip to False to skip Uni-Mol-dependent runs

representations = list(REPS_TIER_1)
if RUN_TIER_2:
    representations += REPS_TIER_2

print(f'Running {len(representations)} representations:')
for r in representations:
    cfg = ALL_REPRESENTATIONS[r]
    print(f'  {r:<16s} {cfg}')

## Section 3 — Getters that reuse cached outputs

Avoid recomputing mol2vec / Uni-Mol embeddings: the existing
`benchmark.ipynb` saved the 300D mol2vec matrices in `output_benchmark/`,
and `benchmark_unimol.ipynb` saved the 512D Uni-Mol matrices in
`output_unimol/`. We pass getter callbacks into `run_grid` so it hydrates
those matrices directly from disk.

In [ ]:
def mol2vec_matrix_getter(case):
    """Load the pre-computed 300D mol2vec matrix for this case, if present."""
    path = os.path.join(MOL2VEC_OUTPUT, f'{case.stem}_300d_vectors.csv')
    if os.path.exists(path):
        return pd.read_csv(path).values.astype(np.float64)
    return None

def unimol_matrix_getter(case):
    """Load the pre-computed 512D Uni-Mol matrix, if present."""
    path = os.path.join(UNIMOL_OUTPUT, f'{case.stem}_unimol_512d_vectors.csv')
    if os.path.exists(path):
        return pd.read_csv(path).values.astype(np.float64)
    return None

for c in CASES:
    m = mol2vec_matrix_getter(c)
    u = unimol_matrix_getter(c)
    tag_m = f'mol2vec {m.shape}' if m is not None else '—'
    tag_u = f'unimol {u.shape}' if u is not None else '—'
    print(f'  {c.stem:<22s} {tag_m:<22s} {tag_u}')

## Section 4 — Run the full grid

`run_grid` iterates over every case, builds each representation (via cache)
and evaluates every compatible metric for every N in `case.n_values`
against every ground-truth property column. The output is a tidy
DataFrame with one row per (case, property, representation, metric, N).

Expected runtime on CPU (cached Uni-Mol): ~20–40 minutes for all 11+ cases.
For an even faster first pass, comment out QM9 (two 127k-row cases).

In [ ]:
INCLUDE_QM9 = True   # set False to skip the two 127k-row QM9 cases

cases_to_run = [c for c in CASES if INCLUDE_QM9 or c.dataset_label != 'QM9']
print(f'Running {len(cases_to_run)} cases with '
      f'{len(representations)} representations')

t0 = time.time()
results = run_grid(
    cases=cases_to_run,
    representations=representations,
    cache_dir=REP_CACHE,
    mol2vec_model=mol2vec_model,
    mol2vec_matrix_getter=mol2vec_matrix_getter,
    unimol_matrix_getter=unimol_matrix_getter,
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
    verbose=True,
)
print(f'\nTOTAL: {len(results):,} rows in {(time.time()-t0)/60:.1f} min')

# Persist raw grid so we can re-render tables / figures later without re-running.
out_path = os.path.join(RESULTS_DIR, 'grid_results.csv')
results.to_csv(out_path, index=False)
print(f'Raw grid saved to {out_path}')

## Section 5 — Summary tables

### 5a — Win rate of every (rep, metric) vs ECFP4/Tanimoto

Win rate = fraction of (case, property, N) tuples where this combination
gave a *lower* mean absolute deviation (i.e. better ground-truth
alignment) than the paper's current baseline (ECFP4 2048 / Tanimoto).

In [ ]:
wr = win_rate_table(results)
wr.to_csv(os.path.join(RESULTS_DIR, 'win_rate_vs_ecfp4_tanimoto.csv'), index=False)
print(f'{len(wr)} (rep, metric) combinations. Top 20:')
wr.head(20)

### 5b — Mean performance by representation family

`mean_norm < 1.0` means the family's average deviation is *lower* than a
random top-N draw — i.e. the method is doing real work.

In [ ]:
fam = summary_by_family(results)
fam.to_csv(os.path.join(RESULTS_DIR, 'summary_by_family.csv'), index=False)
fam

### 5c — MSI (Mahalanobis θ) across representations

Isolates the claim central to the paper: *MSI generalises across
representations*. If $\theta_M$ does better than Tanimoto/cosine on the
same underlying matrix, that is evidence that the **metric** contributes
independently of the **representation**.

In [ ]:
msi_view = (
    results[results['metric'] == 'mahalanobis_theta']
    .groupby('rep')
    .agg(mean_deviation=('deviation', 'mean'),
         mean_norm=('deviation_norm', 'mean'),
         median_p=('p_value', 'median'),
         n_rows=('deviation', 'size'))
    .sort_values('mean_norm')
)
msi_view

## Section 6 — Heatmaps

### 6a — Mean `deviation_norm` over the full grid

One tile per (representation, metric). Darker green = lower normalised
deviation = better property alignment. Cells beyond the compatibility
constraints (e.g. Tanimoto on a continuous descriptor) are hidden.

In [ ]:
pivot = (
    results
    .groupby(['rep', 'metric'])['deviation_norm']
    .mean()
    .unstack('metric')
)

# Mask incompatible cells so we don't give misleading colours
def _family(rep):
    if rep in ('rdkit_physchem',): return 'descriptor'
    if rep in ('mol2vec', 'unimol', 'hybrid'): return 'embedding'
    return 'fingerprint'

mask = pd.DataFrame(False, index=pivot.index, columns=pivot.columns)
for rep in pivot.index:
    allowed = set(COMPATIBLE_METRICS[_family(rep)])
    for m in pivot.columns:
        if m not in allowed:
            mask.loc[rep, m] = True

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.3f', mask=mask,
            cmap='YlGn_r', cbar_kws={'label': 'mean normalised deviation'},
            vmin=0.3, vmax=1.1, ax=ax)
ax.set_title('Mean normalised property deviation across all cases and N — '
             'lower is better (< 1.0 = better than random top-N)')
ax.set_xlabel('similarity metric')
ax.set_ylabel('representation')
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'heatmap_norm_deviation.png'), dpi=200)
plt.show()

### 6b — Breakdown by dataset family

Electronic-property datasets (QM8, QM9) and macroscopic experimental ones
(ESOL, FreeSolv, Lipophilicity) live in different regimes. Plotting them
separately exposes where each representation shines.

In [ ]:
GROUPS = {
    'electronic (QM8, QM9)':   ['QM8', 'QM9'],
    'experimental (ESOL, FreeSolv, Lipophilicity)':
        ['ESOL', 'FreeSolv', 'Lipophilicity'],
}

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, (title, labels) in zip(axes, GROUPS.items()):
    subset = results[results['dataset'].isin(labels)]
    piv = (subset.groupby(['rep', 'metric'])['deviation_norm']
           .mean().unstack('metric'))
    sub_mask = pd.DataFrame(False, index=piv.index, columns=piv.columns)
    for rep in piv.index:
        allowed = set(COMPATIBLE_METRICS[_family(rep)])
        for m in piv.columns:
            if m not in allowed:
                sub_mask.loc[rep, m] = True
    sns.heatmap(piv, annot=True, fmt='.3f', mask=sub_mask,
                cmap='YlGn_r', cbar_kws={'label': 'mean norm deviation'},
                vmin=0.3, vmax=1.1, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('metric'); ax.set_ylabel('representation')

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'heatmap_by_domain.png'), dpi=200)
plt.show()

## Section 7 — Direct answer to each reviewer point

Each block below renders exactly the table/figure cited in the author
response letter. The idea is: the reviewer should be able to verify every
claim we make without having to re-run anything.

### 7.1 — Point 2: Tversky generalises Tanimoto

We include Tversky with two asymmetric weightings. Compare
the three on the **same** ECFP4 matrix to show the metric dimension in
isolation (the representation is identical by construction).

In [ ]:
tvy = (
    results[
        (results['rep'] == 'ecfp4_2048')
        & (results['metric'].isin(
            ['tanimoto', 'dice', 'tversky_8_2', 'tversky_2_8']))
    ]
    .groupby('metric')
    .agg(mean_norm=('deviation_norm', 'mean'),
         median_p=('p_value', 'median'),
         n_rows=('deviation', 'size'))
    .sort_values('mean_norm')
)
print('On the same ECFP4/2048 representation:')
tvy

### 7.2 — Point 3: representation matters (maybe more than metric)

Hold the metric fixed (Tanimoto and $\theta_M$, separately) and vary the
representation. The spread across representations is the "representation
effect"; large spread → choice of representation is the dominant driver.

In [ ]:
print('=== Tanimoto, varying representation ===')
tan = (
    results[(results['metric'] == 'tanimoto')
            & results['rep'].isin(['ecfp4_1024', 'ecfp4_2048', 'ecfp4_4096',
                                   'ecfp6_2048', 'maccs', 'rdkit_fp'])]
    .groupby('rep')
    .agg(mean_norm=('deviation_norm', 'mean'),
         median_p=('p_value', 'median'),
         n_rows=('deviation', 'size'))
    .sort_values('mean_norm')
)
print(tan)

print('\n=== Mahalanobis θ, varying representation ===')
mah = (
    results[results['metric'] == 'mahalanobis_theta']
    .groupby('rep')
    .agg(mean_norm=('deviation_norm', 'mean'),
         median_p=('p_value', 'median'),
         n_rows=('deviation', 'size'))
    .sort_values('mean_norm')
)
print(mah)

### 7.3 — Point 5: wider-representation benchmark

Final, paper-ready table. One row per representation, one column per
metric. Each cell is mean normalised deviation (lower = better).

In [ ]:
final = (
    results
    .groupby(['rep', 'metric'])['deviation_norm']
    .mean()
    .unstack('metric')
    .round(3)
)
final.to_csv(os.path.join(RESULTS_DIR, 'final_representation_x_metric.csv'))
final

## Section 8 — Reviewer point 4: larger / more complex references

Subset the grid to the Lipophilicity cases that use a large (MW ≥ 350)
or structurally complex reference — celecoxib, sildenafil, losartan,
atorvastatin, simvastatin, tamoxifen, fluoxetine, imipramine —
as long as `prepare_benchmark.py` found them in the dataset. This is the
evidence table for Reviewer 1, point 4.

In [ ]:
complex_refs = {'celecoxib', 'sildenafil', 'losartan', 'atorvastatin',
                'simvastatin', 'tamoxifen', 'fluoxetine', 'imipramine',
                'diclofenac', 'naproxen'}

complex_view = results[results['reference'].isin(complex_refs)]
if complex_view.empty:
    print('No complex-reference cases in the grid yet. '
          'Run prepare_benchmark.py to scan Lipophilicity for more drugs.')
else:
    tbl = (
        complex_view
        .groupby(['reference', 'rep', 'metric'])['deviation_norm']
        .mean()
        .unstack('metric')
        .round(3)
    )
    tbl.to_csv(os.path.join(RESULTS_DIR, 'complex_refs_detail.csv'))
    print(f'{complex_view["reference"].nunique()} complex reference molecules in the grid.')
    display(tbl.head(40))

## Section 9 — Next step

The artefacts in `output_grid/` are the inputs for:

- **author response letter** — quote the win-rate table and
  `final_representation_x_metric.csv`;
- **revised manuscript** — add a "Representation × metric" figure and
  a supplementary table built from `grid_results.csv`.

When QM9 is excluded the grid runs in minutes, which makes the Tier-1
sweep cheap enough to re-run every time a new reference molecule is
added to `prepare_benchmark.py`.